In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, matthews_corrcoef
import joblib

In [3]:
df = pd.read_csv("../features/merged_df.csv")
df = df.dropna()
df = df.iloc[:, 2:] 

In [ ]:
y = df['target']             
X = df.drop(columns=["target"]) 

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

knn_model = KNeighborsClassifier(n_neighbors=5)

In [ ]:
cv_scores = cross_val_score(knn_model, X_scaled, y, cv=10, scoring='accuracy')
print(f" 10-Fold CV Scores: {cv_scores}")
print(f" Mean CV Accuracy: {cv_scores.mean():.4f}")

✅ 10-Fold CV Scores: [0.66037736 0.69230769 0.76923077 0.63461538 0.67307692 0.63461538
 0.69230769 0.57692308 0.65384615 0.61538462]
✅ Mean CV Accuracy: 0.6603


In [ ]:
knn_model.fit(X_scaled, y)
y_pred_knn = knn_model.predict(X_scaled)

cm = confusion_matrix(y, y_pred_knn, labels=knn_model.classes_)
print(" Confusion Matrix:")
print(cm)

In [ ]:
sensitivity_list = []
specificity_list = []
mcc_list = []
accuracy_list = []

for i in range(len(knn_model.classes_)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)

    # Sensitivity = TP / (TP + FN)
    sensitivity = (TP / (TP + FN)) * 100 if (TP + FN) > 0 else 0

    # Specificity = TN / (TN + FP)
    specificity = (TN / (TN + FP)) * 100 if (TN + FP) > 0 else 0

    # MCC = (TP * TN - FP * FN) / sqrt(...)
    mcc_numerator = (TP * TN) - (FP * FN)
    mcc_denominator = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = (mcc_numerator / mcc_denominator) * 100 if mcc_denominator > 0 else 0

    # Accuracy = (TP + TN) / Total
    accuracy = ((TP + TN) / (TP + FN + FP + TN)) * 100


    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)
    mcc_list.append(mcc)
    accuracy_list.append(accuracy)

In [14]:
print("\n📊 KNN Classification Report (per class):")
print("=" * 60)
print(f"{'Class':<10}{'Sensitivity':>15}{'Specificity':>15}{'MCC':>10}{'Accuracy':>12}")
print("-" * 60)

for idx, cls in enumerate(knn_model.classes_):
    print(f"{str(cls):<10}{sensitivity_list[idx]:>14.2f}%{specificity_list[idx]:>14.2f}%"
          f"{mcc_list[idx]:>10.2f}%{accuracy_list[idx]:>11.2f}%")

print("=" * 60)



📊 KNN Classification Report (per class):
Class         Sensitivity    Specificity       MCC    Accuracy
------------------------------------------------------------
Fold1              75.72%         90.52%     67.17%      85.60%
Fold2              16.67%        100.00%     40.23%      97.12%
Fold3              86.54%         87.86%     73.86%      87.33%
Fold4              74.59%         88.72%     61.05%      85.41%


In [15]:
overall_accuracy = accuracy_score(y, y_pred_knn) * 100
print(f" Overall KNN Accuracy: {overall_accuracy:.2f}%")
print(f" Average Accuracy (across classes): {np.mean(accuracy_list):.2f}%")


 Overall KNN Accuracy: 77.74%
 Average Accuracy (across classes): 88.87%


In [17]:
joblib.dump(knn_model, '../models/knn_final_model.pkl')
print(" KNN model saved as '../models/knn_final_model.pkl'")

 KNN model saved as 'knn_final_model.pkl'
